# SpaceObject Module - Complete API Overview & Examples
---
> *Created by Henry Flushman — Updated 2026-02-17*

## Table of Contents
1. [Overview](#overview)
2. [ObjectType Enum](#objecttype-enum)
3. [SpaceObject Class](#spaceobject-class)
4. [Subclasses](#subclasses)
5. [SpaceTrackJSONParser](#spacetracjsonparser)
6. [SpaceObjectList](#spaceobjectlist)
7. [Practical Examples](#practical-examples)

---

## Overview

The `SpaceObject` module provides intelligent wrappers around Space-Track / SATCAT-like records. It handles:
- Smart extraction and normalization of space object data from JSON records
- Multiple object types (Payload, Debris, Rocket Body, Unknown) via typed subclasses
- Flexible construction: raw record mode (from SATCAT JSON) or manual configuration
- Comprehensive filtering and querying via `SpaceObjectList` container
- TLE (Two-Line Element) handling and orbital parameter extraction
- Internal storage using private attributes to avoid property recursion
- Decay status tracking (decayed vs. active in-orbit objects)
- Maneuverability assessment (only Payloads are considered maneuverable)

**Design Philosophy:**
- Normalize many variant field names from different SATCAT sources
- Support two modes: raw records (JSON dicts) and manual construction
- Expose public properties for backward compatibility
- Store full record so ALL fields are preserved
- Extract common fields into convenient attributes
- Handle multiple object constructions efficiently

---

## ObjectType Enum

Internal enumeration for normalized object type representation.

**Members:**
- `PAYLOAD` - "PAYLOAD"
- `DEBRIS` - "DEBRIS"
- `ROCKET_BODY` - "ROCKET BODY"
- `UNKNOWN` - "UNKNOWN"

Used internally by SpaceObject to standardize type handling and normalization.

---

## SpaceObject Class

### Constructor

```python
SpaceObject(
    rawRecord: Optional[Dict[str, Any]] = None,
    objectType: Optional[str] = None,
    name: Optional[str] = None,
    noradID: Optional[str] = None,
    objectID: Optional[str] = None,
    origin: Optional[str] = None,
    launchDate: Optional[str] = None,
    epoch: Optional[str] = None,
    tle_line1: Optional[str] = None,
    tle_line2: Optional[str] = None,
)
```

**Two Modes:**
- **Raw Record Mode**: Pass `rawRecord` dict; fields are extracted from known SATCAT field names
- **Manual Mode**: Set `rawRecord=None`; provide fields via kwargs

### Public Properties

| Property | Type | Description |
|----------|------|-------------|
| `noradID` | str | NORAD catalog identifier |
| `objectID` | str | International designator |
| `name` | str | Satellite/object name |
| `origin` | str | Country/origin code |
| `launchDate` | date | Launch date as datetime.date |
| `epoch` | date | Epoch date as datetime.date |
| `tle_line1` | str | First TLE line |
| `tle_line2` | str | Second TLE line |
| `rcs` | Any | Radar Cross Section value |
| `rcs_size` | str | RCS size category |
| `objectType` | str | Normalized object type string |
| `apogee` | float | Apogee altitude (km) |
| `perigee` | float | Perigee altitude (km) |
| `inclination` | float | Orbital inclination (degrees) |
| `period` | float | Orbital period (minutes) |
| `tle` | str | TLE lines joined by newline |
| `country` | str | Alias for origin |
| `decayDate` | date | Decay/reentry date |
| `orbitSummary` | dict | Dict of apogee, perigee, inclination, period |

### Key Methods

#### Field Access
- `getField(fieldName: str, default=None)` - Get raw field from record
- `listFields()` - List all available field keys
- `get_str(*field_names, default=None)` - Get first non-empty field as string
- `get_float(*field_names, default=None)` - Get first field parseable as float
- `get_int(*field_names, default=None)` - Get first field parseable as int
- `get_date(*field_names, default=None)` - Parse first field into datetime.date

#### Object Type & TLE
- `getObjectType()` - Return normalized type ("PAYLOAD", "DEBRIS", "ROCKET BODY", "UNKNOWN")
- `hasTLE()` - Return True if both TLE lines present
- `getTLE()` - Return TLE lines joined by newline

#### Orbital Parameters
- `getApogee()`, `getPerigee()`, `getInclination()`, `getPeriod()` - Float orbital values
- `getRCSValue()`, `getRCSSize()` - RCS information
- `getOrbitSummary()` - Dict of all orbital parameters

#### Identification & Dates
- `getNORAD()`, `getObjectID()`, `getName()`, `getCountry()` - Identification
- `getLaunchDate()`, `getDecayDate()`, `getEpochDate()` - Date information
- `isCurrent()` - Parse CURRENT field to boolean
- `isDecayed()` - Return True if object has decayed (has decay date)
- `isManeuverable()` - Return True only for Payloads

#### Factory Methods (Class Methods)
- `fromSpaceTrack(rawRecord: Dict[str, Any])` - Create typed subclass from record
- `fromSpaceTrackBatch(rawData: List[Dict])` - Create list of SpaceObjects from records

#### Static Filters
- `filterByType(objects, obj_type)` - Filter by object type
- `filterDecayed(objects)` - Filter objects that have decayed (have decay dates)
- `filterNotDecayed(objects)` - Filter objects that have NOT decayed (no decay dates)
- `onlyPayloads()`, `onlyDebris()`, `onlyRocketBodies()`, `onlyUnknown()` - Type-specific filters

---

## Subclasses

Convenience subclasses that automatically set the object type.

```python
class Payload(SpaceObject):          # objectType="PAYLOAD"
class Debris(SpaceObject):           # objectType="DEBRIS"
class RocketBody(SpaceObject):       # objectType="ROCKET BODY"
class Unknown(SpaceObject):          # objectType="UNKNOWN"
```

All inherit SpaceObject functionality with predefined type.

---

## SpaceTrackJSONParser

Helper class for parsing Space-Track JSON responses.

### Static Methods
- `parse(payload)` - Parse bytes/str/dict/list into List[SpaceObject]
  - Auto-detects format; decodes bytes; parses JSON strings
- `parseByType(payload, obj_type)` - Parse then filter by normalized type

---

## SpaceObjectList

Container for SpaceObject instances with convenient filtering and utility methods.

### Constructor & Class Methods
- `__init__(objects: Optional[List[SpaceObject]])` - Initialize with objects
- `from_payload(payload)` - Parse JSON/bytes/str/dict/list and return SpaceObjectList
- `from_list(objects: List[SpaceObject])` - Create from existing list

### Container Methods
- `__len__()`, `__iter__()`, `__getitem__(idx)` - Standard container protocol
- `to_list()` - Return shallow copy of internal list
- `append(obj)`, `extend(objs)` - Add objects

### Filtering Methods
- `filter_by_type(obj_type)` - Filter by object type
- `filter_decayed()` - Filter objects that have decayed
- `filter_not_decayed()` - Filter objects that have NOT decayed
- `only_payloads()`, `only_debris()`, `only_rocket_bodies()`, `only_unknown()` - Type shortcuts
- `filter(**criteria)` - General-purpose filter supporting:
  - String: case-insensitive exact match
  - Collection: membership test
  - None: match only None values
  - Callable: predicate function (value → bool)

### Utility Methods
- `group_by_type()` - Group objects by normalized type → Dict[str, SpaceObjectList]
- `sort_by(attr: str, reverse=False)` - Sort by attribute (property or raw field)
- `map(fn: Callable)` - Apply function to each object → List[Any]

---

## Complete API Quick Reference

### Constructor Variants
```python
# From SATCAT record
obj = SpaceObject(rawRecord=satcat_dict)

# From factory (returns appropriate subclass)
obj = SpaceObject.fromSpaceTrack(satcat_dict)

# From subclass
obj = Payload(rawRecord=satcat_dict)

# Manual construction
obj = SpaceObject(rawRecord=None, name="SAT", noradID="123", objectType="PAYLOAD")
```

### Filtering Examples
```python
# Type-based
payloads = sol.only_payloads()
decayed = sol.filter_decayed()
not_decayed = sol.filter_not_decayed()
active_payloads = sol.only_payloads().filter_not_decayed()

# General-purpose
cis_objs = sol.filter(origin="CIS")  # String match
high_ap = sol.filter(apogee=lambda v: v is not None and float(v) > 800)  # Predicate
multi_country = sol.filter(origin=["USA", "CIS", "ESA"])  # Membership
```

### Status Check Examples
```python
# Individual object checks
if obj.isDecayed():
    print(f"{obj.name} has decayed")
if obj.isManeuverable():
    print(f"{obj.name} is a maneuverable payload")

# Collection-based checks
maneuverable_payloads = sol.only_payloads().map(lambda o: o if o.isManeuverable() else None)
active_and_maneuverable = sol.filter_not_decayed().only_payloads()
```

### Data Access Examples
```python
# Properties
name = obj.name
apogee = obj.apogee  # Computed from getter

# Methods
orbit = obj.getOrbitSummary()  # Dict with all orbital params
tle = obj.getTLE()  # TLE lines or None
decay = obj.getDecayDate()  # datetime.date or None

# Raw fields
value = obj.getField("CUSTOM_FIELD")
```

# Practical Examples

## Setup and File Verification

In [ ]:
from pathlib import Path
import json

# Point these to wherever your files live
MODULE_PATH = Path("SpaceObject_class.py")
SATCAT_JSON_PATH = Path("satcat_test.json")
print("Module exists:", MODULE_PATH.exists(), "->", MODULE_PATH.resolve())
print("JSON exists  :", SATCAT_JSON_PATH.exists(), "->", SATCAT_JSON_PATH.resolve())

In [ ]:
# Import the API including the new SpaceObjectList
from SpaceObject_class import (
    SpaceObject,
    Payload,
    Debris,
    RocketBody,
    Unknown,
    SpaceTrackJSONParser,
    SpaceObjectList,
)

## Example 1: Load & Parse SATCAT JSON

In [ ]:
# Load the JSON file (list[dict])
satcat_records = json.load(open(SATCAT_JSON_PATH, "r", encoding="utf-8"))
print(f"Type of python variable: {type(satcat_records)}")
print(f"Length of list: {len(satcat_records)}")

# Parse into SpaceObject instances using the batch factory and wrap into a SpaceObjectList
objects = SpaceObject.fromSpaceTrackBatch(satcat_records)
sol = SpaceObjectList.from_list(objects)

print("Loaded objects into SpaceObjectList:", len(sol))
print("First object  :", sol[0])
print("Python class  :", type(sol[0]).__name__)

## Example 2: Access Object Properties and Fields

In [ ]:
obj = sol[0]

# Basic printing / repr
print(obj)

# Raw field access
print("OBJECT_NAME:", obj.getField("OBJECT_NAME"))
print("A FIELD THAT DOESN'T EXIST (default='N/A'):", obj.getField("DOES_NOT_EXIST", default="N/A"))

# Field / property inspection
fields = obj.listFields()
print("Number of fields:", len(fields))
print("First 10 fields:", fields[:10])
print("getObjectType():", obj.getObjectType())

# Example properties
print(f"              Apogee:   {obj.apogee}")
print(f"             Perigee:   {obj.perigee}")
print(f"         Inclination:   {obj.inclination}")
print(f"NORAD Identification:   {obj.noradID}")
print(f"              Origin:   {obj.origin}")
print(f"         Launch Date:   {obj.launchDate}")
print(f"          Decay Date:   {obj.decayDate}")

## Example 3: Check Decay Status and Maneuverability

In [ ]:
# Check decay status for objects
print("Decay Status Check:")
for i in range(min(5, len(sol))):
    obj = sol[i]
    status = "DECAYED" if obj.isDecayed() else "IN ORBIT"
    maneuverable = "(Maneuverable)" if obj.isManeuverable() else ""
    print(f"  {obj.name:30s} | {obj.getObjectType():12s} | {status:10s} {maneuverable}")

## Example 4: Create Objects Manually

In [ ]:
my_record = {
    "OBJECT_NAME": "MY_TEST_SAT",
    "OBJECT_ID": "2099-001A",
    "NORAD_CAT_ID": "99999",
    "OBJECT_TYPE": "PAYLOAD",
    "APOGEE": 550.0,
    "PERIGEE": 540.0,
    "INCLINATION": 97.6,
}

my_obj = SpaceObject.fromSpaceTrack(my_record)
print("Factory-built:", my_obj, "| class:", type(my_obj).__name__)

my_payload = Payload(rawRecord=my_record)
print("Direct subclass:", my_payload, "| class:", type(my_payload).__name__)

generic = SpaceObject(rawRecord=my_record)
print("Generic:", generic, "| class:", type(generic).__name__)

## Example 5: Filter by Type and Decay Status

In [ ]:
# Use the SpaceObjectList helpers
sol_all = sol  # already a SpaceObjectList
print("All objects   :", len(sol_all))

# Filter by type
rocket_bodies = sol_all.only_rocket_bodies()
payloads = sol_all.only_payloads()
print("Rocket bodies :", len(rocket_bodies))
print("Payloads      :", len(payloads))

# Filter by decay status
decayed = sol_all.filter_decayed()
not_decayed = sol_all.filter_not_decayed()
print("Decayed       :", len(decayed))
print("Not Decayed   :", len(not_decayed))

# Chain filters: active (not decayed) payloads
active_payloads = sol_all.only_payloads().filter_not_decayed()
print("Active Payloads:", len(active_payloads))

# Chain filters: maneuverable objects (payloads that haven't decayed)
maneuverable = sol_all.only_payloads().filter_not_decayed()
print("Maneuverable  :", len(maneuverable))

## Example 6: General-Purpose Filtering

In [ ]:
# Find objects whose name contains 'SL-1' using a callable predicate
sl1_candidates = sol_all.filter(name=lambda v: v is not None and "SL-1" in v)
print("SL-1 matches:", len(sl1_candidates))

# Find objects launched by country 'CIS' (case-insensitive exact match on origin/country)
cis_objs = sol_all.filter(origin="CIS")
print("CIS objects:", len(cis_objs))

# Numeric predicate example: apogee > 800 km
high_apogee = sol_all.filter(apogee=lambda v: v is not None and float(v) > 800)
print("High apogee (>800km):", len(high_apogee))

## Example 7: Advanced Decay Analysis

In [ ]:
# Analyze decay statistics by type
print("Decay Statistics by Type:")
print("=" * 60)

for obj_type in ["PAYLOAD", "DEBRIS", "ROCKET BODY", "UNKNOWN"]:
    type_objs = sol_all.filter_by_type(obj_type)
    decayed_count = len(type_objs.filter_decayed())
    not_decayed_count = len(type_objs.filter_not_decayed())
    total = len(type_objs)
    
    if total > 0:
        decay_pct = 100 * decayed_count / total
        print(f"{obj_type:12s}: {total:3d} total | {decayed_count:3d} decayed ({decay_pct:5.1f}%) | {not_decayed_count:3d} active")

## Example 8: MiniROVER Danger Checker (Updated)

In [ ]:
# Select a reference object (must not be decayed)
not_decayed_objs = sol_all.filter_not_decayed()
if len(not_decayed_objs) == 0:
    print("No active (non-decayed) objects available for hazard detection.")
else:
    # Use first active object as reference
    ref = not_decayed_objs[0]
    ref_ap = float(ref.getField("APOGEE")) if ref.getField("APOGEE") else None
    ref_pe = float(ref.getField("PERIGEE")) if ref.getField("PERIGEE") else None
    ref_inc = float(ref.getField("INCLINATION")) if ref.getField("INCLINATION") else None

    if ref_ap and ref_pe and ref_inc:
        print("MiniROVER Reference Orbit (Active Object):")
        print(f"  Name:         {ref.name}")
        print(f"  NORAD ID:     {ref.noradID}")
        print(f"  Type:         {ref.getObjectType()}")
        print(f"  Decayed:      {ref.isDecayed()}")
        print(f"  Maneuverable: {ref.isManeuverable()}")
        print(f"  Apogee:       {ref_ap:.1f} km")
        print(f"  Perigee:      {ref_pe:.1f} km")
        print(f"  Inclination:  {ref_inc:.2f}°")

        # Define similarity thresholds for "close approach" risk
        APOGEE_TOL_KM = 25.0
        PERIGEE_TOL_KM = 25.0
        INC_TOL_DEG = 1.0

        # Find potential hazards (only from active objects)
        matches = []
        for o in not_decayed_objs:
            try:
                ap = float(o.getField("APOGEE"))
                pe = float(o.getField("PERIGEE"))
                inc = float(o.getField("INCLINATION"))
            except Exception:
                continue

            if abs(ap - ref_ap) <= APOGEE_TOL_KM and abs(pe - ref_pe) <= PERIGEE_TOL_KM and abs(inc - ref_inc) <= INC_TOL_DEG:
                # score: smaller is "closer"
                score = abs(ap - ref_ap) + abs(pe - ref_pe) + 10.0 * abs(inc - ref_inc)
                matches.append((score, o))

        matches.sort(key=lambda t: t[0])

        print(f"\nPotential hazards (closest first): {len(matches)}")
        for score, o in matches[:10]:
            print(f"score={score:8.3f} | {o.name:30s} | {o.getObjectType():12s} | Decayed: {o.isDecayed()}")
    else:
        print("Reference object missing orbital parameters.")

## Notes

- If your JSON contains `null` values, they become `None` in Python.
- Many SATCAT records have missing orbital elements (especially decayed/historical objects).
- Use `SpaceObjectList.filter()` and `filter_decayed()` / `filter_not_decayed()` / `only_*()` helpers for concise workflows.
- **Decay Status**: Objects with a `DECAY_DATE` field are considered decayed. Use `isDecayed()` to check individual objects.
- **Maneuverability**: Only Payload objects return `True` for `isManeuverable()`. Debris, Rocket Bodies, and Unknown objects return `False`.
- SpaceObjectList methods are chainable: `sol.only_payloads().filter_not_decayed().sort_by('name')`
- Filters return new SpaceObjectList instances, not modifying the original.
- The `filter()` method tries object properties first, then falls back to raw record fields.
- For complete API documentation, see `SpaceObject_API_Reference.md`